# Quickstart — score a real clip with `tone-metric`

A 2-minute **functional** test (not just an import check): it installs the package, runs the
reference-free `tone_i2` metric on one real audio clip + its tone-marked transcript, then proves the
metric is doing its job by **flattening the pitch** (PSOLA) and showing `tone_i2` drops while the words
are unchanged. Both clips play inline.

**Runtime:** CPU is fine (the default path needs no torch). ~2 min.

You provide a short **Yoruba** clip and its **tone-marked** transcript (upload, or pull one from S3).

## 1 · Install the package (this is also the install smoke-test)

In [ ]:
%pip install --quiet "git+https://github.com/mosesdaudu001/tone-on-a-budget.git" swift-f0
# core deps (numpy/scipy/librosa/soundfile/praat-parselmouth/pyworld) come with the package;
# swift-f0 is the F0 backend. No torch needed for the default (proportional-window) path.
import tone_metric
print("OK — tone_metric", tone_metric.__version__)

## 2 · Provide a clip + its TONE-MARKED transcript

`TEXT` **must carry the tone diacritics** (´ = High, ` = Low, unmarked = Mid) — that is the answer key
the metric scores against. Set `SOURCE`:
- `"upload"` — pick a short `.wav` from your machine (Colab file picker).
- `"s3"` — pull one clip from the project bucket (needs `AWS_*` Colab secrets); edit `S3_KEY`.

In [ ]:
import librosa, IPython.display as ipd

SOURCE = "upload"                       # "upload"  or  "s3"
TEXT   = "Mo rí ọkọ̀ ní ọjà."          # <-- tone-marked transcript of YOUR clip (EDIT THIS)
S3_KEY = "tts_data/yoruba/omnivoice_probe/REPLACE_WITH_A_REAL_CLIP.wav"   # used only if SOURCE=="s3"

if SOURCE == "upload":
    from google.colab import files
    up = files.upload()                 # choose one short Yoruba .wav
    WAV_PATH = next(iter(up))
elif SOURCE == "s3":
    import boto3
    from google.colab import userdata
    s3 = boto3.client("s3",
        aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY"))
    WAV_PATH = "/tmp/quickstart_clip.wav"
    s3.download_file("codec-audio-data", S3_KEY, WAV_PATH)
else:
    raise ValueError("SOURCE must be 'upload' or 's3'")

wav, sr = librosa.load(WAV_PATH, sr=24000, mono=True)
print(f"loaded {WAV_PATH}  |  {len(wav)/sr:.2f}s @ {sr}Hz  |  text: {TEXT}")
ipd.display(ipd.Audio(wav, rate=sr))

## 3 · Score it — and prove the metric responds to tone

`tone_i2` (`accuracy`) is balanced H/M/L tone-classification accuracy in `[0.333 .. 1.0]` — **always read
it together with `coverage`** (it is `NaN` when nothing is scoreable, e.g. a clip too short or unvoiced).

In [ ]:
from tone_metric import f0_abs_score, psola_flatten

def show(label, r):
    a = r["accuracy"]
    a = f"{a:.3f}" if a == a else "NaN (nothing scoreable)"     # a==a is False for NaN
    print(f"{label:20s} tone_i2={a}  coverage={r['coverage']:.2f}  "
          f"scored={r['n_scored']}/{r['n_tbu']} TBUs  method={r['method']}  backend={r['backend']}")

orig = f0_abs_score(wav, sr, TEXT)                 # tone_i2 on the real clip
flat_wav = psola_flatten(wav, sr)                  # destroy pitch, keep the words
flat = f0_abs_score(flat_wav, sr, TEXT)            # tone_i2 after flattening

show("original", orig)
show("pitch-flattened", flat)
if orig["accuracy"] == orig["accuracy"] and flat["accuracy"] == flat["accuracy"]:
    d = orig["accuracy"] - flat["accuracy"]
    print(f"\n--> tone_i2 drop when pitch is flattened: {d:+.3f}  "
          f"({'GOOD — the metric sees tone' if d > 0 else 'no drop (try a longer/clearer clip or the §4 ASR path)'})")

print("\noriginal:");        ipd.display(ipd.Audio(wav, rate=sr))
print("pitch-flattened:");    ipd.display(ipd.Audio(flat_wav, rate=sr))

## 4 · (Optional) the accurate path — forced alignment with MMS

The default above uses **proportional** windows (no ASR). For the exact metric used in the poster, give
`f0_abs_score` an MMS-yor CTC model for **forced alignment**. Heavier (downloads MMS, needs torch).

In [ ]:
# %pip install --quiet torch transformers torchaudio
from transformers import Wav2Vec2ForCTC, AutoProcessor
import torch

proc = AutoProcessor.from_pretrained("facebook/mms-1b-all")
proc.tokenizer.set_target_lang("yor")
asr = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="yor",
                                     ignore_mismatched_sizes=True).eval()

orig_fa = f0_abs_score(wav, sr, TEXT, asr=asr, proc=proc, device="cpu")
flat_fa = f0_abs_score(flat_wav, sr, TEXT, asr=asr, proc=proc, device="cpu")
show("original (forced)", orig_fa)
show("flattened (forced)", flat_fa)

## What this shows

If §3 printed a `tone_i2` for the original clip (with non-zero `coverage`) **and** a lower one for the
flattened clip, the package is installed and the metric is working end-to-end on real audio. For the
full study (synthesis, the data-efficiency sweep, the CER-is-tone-blind demo) see `01`–`09` and the
repo README.